# 08 — 2-Step, Agentic & Hybrid RAG

## Learning requirements
Phân biệt ba architecture:

### 2-Step RAG
`Query -> Retrieve -> Generate`

Ưu: predictable, low latency, dễ evaluate.

### Agentic RAG
`Query -> Agent -> decides whether/what to retrieve -> Agent`

Ưu: flexible; nhược: trajectory/cost khó predict hơn.

### Hybrid RAG
Có deterministic stages như rewrite/validate/rerank kết hợp agentic decisions.

In [ ]:
# Assume `vectorstore` is created as in Notebook 07.
# This cell is intentionally standalone in structure: adapt to your persisted DB.
from langchain_core.prompts import ChatPromptTemplate
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))
from src.providers import get_chat_model

model = get_chat_model()

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer only from the supplied context. "
     "If context is insufficient, say you do not have enough evidence."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

In [ ]:
def two_step_rag(question: str, vectorstore, k: int = 4):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(
        f"[{d.metadata.get('source', 'unknown')}] {d.page_content}"
        for d in docs
    )
    response = model.invoke(RAG_PROMPT.format_messages(
        question=question,
        context=context,
    ))
    return {"answer": response.content, "documents": docs}

In [ ]:
# Retriever-as-tool for agentic RAG.
from langchain.tools import tool
from langchain.agents import create_agent

def build_agentic_rag(vectorstore):
    @tool
    def search_knowledge_base(query: str) -> list[dict]:
        """Search internal project documentation for evidence relevant to the query."""
        docs = vectorstore.similarity_search(query, k=4)
        return [
            {"content": d.page_content, "metadata": d.metadata}
            for d in docs
        ]

    return create_agent(
        model=model,
        tools=[search_knowledge_base],
        system_prompt=(
            "Use the knowledge-base tool when the question depends on project facts. "
            "Cite source metadata from tool results."
        ),
    )

## Hybrid design exercise

Implement graph/pipeline:

```text
classify query
    |
    +-- no retrieval -> answer
    |
    +-- retrieval
         -> rewrite query
         -> retrieve
         -> grade evidence
         -> if weak: rewrite once
         -> generate
         -> citation check
```

Đặt hard limit cho rewrite loop.

## Required output — `artifacts/rag-benchmark.md`

So sánh 2-step vs agentic vs hybrid trên cùng dataset:
- answer correctness;
- retrieval hit rate;
- citation correctness;
- latency;
- model/tool calls;
- token/cost estimate.

## Done criteria
Bạn chọn RAG architecture dựa trên requirement/metrics thay vì “agentic luôn tốt hơn”.